# Reinforcement learning: tabular Q-learning on FrozenLake

**Reinforcement learning** agents learn from **rewards** through interaction. Here we use **tabular Q-learning** (small discrete state space) on Gymnasium's `FrozenLake-v1`.

- **States:** grid cells (encoded as integers in the default observation).
- **Actions:** {Left, Down, Right, Up}.
- **Goal:** reach the goal without falling in a hole; slippery ice makes transitions stochastic when `is_slippery=True`.

In [ ]:
import numpy as np
import gymnasium as gym

env = gym.make("FrozenLake-v1", is_slippery=True)
n_states = env.observation_space.n
n_actions = env.action_space.n
Q = np.zeros((n_states, n_actions))

alpha = 0.5
gamma = 0.99
epsilon_start, epsilon_end = 0.9, 0.05
n_episodes = 8000

rng = np.random.default_rng(0)

for ep in range(n_episodes):
    eps = epsilon_start + (epsilon_end - epsilon_start) * ep / n_episodes
    state, _ = env.reset()
    done = False
    while not done:
        if rng.random() < eps:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[state]))
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        td_target = reward + gamma * float(np.max(Q[next_state])) * (0.0 if done else 1.0)
        td_err = td_target - Q[state, action]
        Q[state, action] += alpha * td_err
        state = next_state

policy = np.argmax(Q, axis=1)

# Evaluate greedy policy
wins = 0
n_eval = 500
for _ in range(n_eval):
    state, _ = env.reset()
    done = False
    while not done:
        action = int(policy[state])
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        if reward > 0:
            wins += 1
            break

print("Success rate (greedy eval):", round(wins / n_eval, 3))
env.close()

## Notes

- FrozenLake is tiny; **tabular Q-learning** is appropriate. Neural nets (DQN) replace the Q table when states are large or continuous.
- With slippery ice, optimal policies are **stochastic**; a single greedy policy may still work after enough training.

## Try this

- Set `is_slippery=False` and compare learning speed.
- Log episode return during training and plot a moving average.